# 04 – DBSCAN (M2)

Phụ trách: M2 (Chu Thành Tân). Theo checklist M2, Phần 4:
- **4.1 Đồ thị k-distance có trọng số**, rút ra các eps ứng viên.
- **4.2 So sánh chạy trên từng chuyến (`full`) với chạy có trọng số (`weighted`)**, đo runtime và RAM.
- 4.3 Lưới eps × min_samples (sẽ bổ sung).
- 4.4 Chọn cấu hình, ghi log, vẽ bản đồ (sẽ bổ sung).

Cách chạy đã được duyệt, khác spec (`docs/Phan 4/4.0-quyet-dinh-dbscan.md`):
1. Fit trên toạ độ duy nhất với `sample_weight` = số chuyến.
2. `min_samples` theo tỷ lệ số chuyến của lát cắt, kèm 5 và 20 của spec để đối chiếu.
3. Eps 250–2500 m, tìm điểm gãy trên k-distance có trọng số.

Dữ liệu đang dùng: **mẫu pilot tạm của M2**. Đổi `DATA_KEY = "model_input"` khi có dữ liệu của M1.

In [ ]:
# ===== O THIET LAP (chep nguyen o nay vao dau moi notebook) =====
# Colab: clone repo, cai thu vien, them goc repo vao sys.path.
# May ca nhan: tu tim goc repo, khong cai gi them.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/tanbentam/KhaiPhaDuLieu.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/KhaiPhaDuLieu")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    os.chdir(ROOT)
else:
    here = pathlib.Path.cwd().resolve()
    ROOT = next(p for p in (here, *here.parents) if (p / "config" / "experiment.yaml").is_file())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Du lieu that tren Colab: bo comment 2 dong duoi khi nhom da chot thu muc Drive
# from src.colab import link_drive_data
# link_drive_data("KhaiPhaDuLieu")

print("Colab:", IN_COLAB, "| Goc repo:", ROOT)

## Tham số

In [ ]:
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from src.config import load_config, resolve_path
from src.models.kmeans import aggregate_unique
from src.models.dbscan import weighted_k_distance, k_distance_curve, min_samples_from_frac, knee_eps

cfg = load_config()
DATA_KEY = "pilot_m2"            # doi thanh "model_input" khi co du lieu cua M1
MANIFEST_KEY = "pilot_m2_manifest" if DATA_KEY == "pilot_m2" else "manifest"
SLICES = list(cfg["time_slices"])
DB = cfg["dbscan"]
df = pd.read_parquet(resolve_path(cfg, DATA_KEY))
SAMPLE_ID = json.loads(resolve_path(cfg, MANIFEST_KEY).read_text(encoding="utf-8"))["sample_id"]

def k_list(n_trips):
    """(nhan, k): k theo spec va k theo ty le so chuyen cua lat cat."""
    ks = [(f"spec {m}", m) for m in DB["min_samples"]]
    ks += [(f"{f * 100:g}% = {min_samples_from_frac(n_trips, f)}", min_samples_from_frac(n_trips, f))
           for f in DB["min_samples_frac"]]
    return ks

for s in SLICES:
    n = int((df.time_slice == s).sum())
    print(f"{s:11s} {n:>6,} chuyen -> k:", [k for _, k in k_list(n)])

## 4.1 Khoảng cách giữa các toạ độ

Khoảng cách tới toạ độ gần nhất cho biết eps phải lớn khoảng bao nhiêu thì các toạ độ mới bắt đầu nối với nhau.

In [ ]:
spacing = []
for s in SLICES:
    Xu, w, _ = aggregate_unique(df.loc[df.time_slice == s, ["x_m", "y_m"]].to_numpy("float64"))
    nn = NearestNeighbors(n_neighbors=2).fit(Xu).kneighbors(Xu)[0][:, 1]
    spacing.append({"time_slice": s, "n_coords": len(Xu), "nn_p10_m": np.percentile(nn, 10), "nn_median_m": np.median(nn),
                    "nn_p90_m": np.percentile(nn, 90), "nn_max_m": nn.max()})
pd.DataFrame(spacing).round(0)

## 4.1 Đồ thị k-distance có trọng số

Với mỗi toạ độ, k-distance có trọng số là **khoảng cách nhỏ nhất cần đi để gom đủ k chuyến** (tính cả chính nó). Đó chính là ngưỡng eps để toạ độ đó trở thành core. Đồ thị vẽ theo từng chuyến: trục ngang là tỷ lệ chuyến tích luỹ, trục dọc là k-distance.

- `self_core_pct`: % chuyến ở toạ độ có ≥ k chuyến, tức là tự là core ở mọi eps.
- `p50/p75/p90`: eps cần để 50/75/90% chuyến nằm ở toạ độ core.
- `knee_eps`: điểm gãy gợi ý, tính trên phần khoảng cách > 0.

In [ ]:
rows, curves = [], {}
for s in SLICES:
    X = df.loc[df.time_slice == s, ["x_m", "y_m"]].to_numpy("float64")
    Xu, w, _ = aggregate_unique(X)
    for label, k in k_list(len(X)):
        share, d = k_distance_curve(X, k)
        curves[(s, label)] = (share, d)
        kd = weighted_k_distance(Xu, w, k)
        pct = lambda q: float(d[np.searchsorted(share, q)])
        try:
            knee = knee_eps(share, d)
        except ValueError:
            knee = np.nan
        rows.append({"time_slice": s, "k_label": label, "k": k, "n_trips": len(X),
                     "self_core_pct": round(w[kd == 0].sum() / w.sum() * 100, 1),
                     "p50_m": pct(0.5), "p75_m": pct(0.75), "p90_m": pct(0.9), "knee_eps_m": knee})
kdist = pd.DataFrame(rows).assign(sample_id=SAMPLE_ID)
kd_path = resolve_path(cfg, "run_log").parent / f"dbscan_kdist_{SAMPLE_ID}.csv"
kdist.to_csv(kd_path, index=False)
print("Da luu:", kd_path.relative_to(cfg["_root"]))
kdist.drop(columns=["sample_id", "n_trips"]).round(0)

In [ ]:
fig, axes = plt.subplots(1, len(SLICES), figsize=(5.2 * len(SLICES), 4.2), sharey=True)
for ax, s in zip(axes, SLICES):
    for (ss, label), (share, d) in curves.items():
        if ss != s: continue
        ls = "--" if label.startswith("spec") else "-"
        ax.step(share * 100, d, where="post", ls=ls, lw=1.4, label=f"k {label}")
        knee = kdist.loc[(kdist.time_slice == s) & (kdist.k_label == label), "knee_eps_m"].iloc[0]
        if np.isfinite(knee):
            ax.scatter([share[np.searchsorted(d, knee)] * 100], [knee], s=18, zorder=3)
    for e in DB["eps_m"]:
        ax.axhline(e, color="grey", lw=.5, alpha=.4)
    ax.set_title(f"{s} (n={int((df.time_slice == s).sum()):,})")
    # 79-100% chuyen o toa do tu la core (k-distance = 0): phong to phan con lai
    x0 = max(0, kdist.loc[kdist.time_slice == s, "self_core_pct"].min() - 3)
    ax.set_xlim(x0, 100.3)
    ax.set_xlabel(f"Ty le chuyen tich luy (%) - phong to tu {x0:.0f}%"); ax.grid(alpha=.3); ax.set_ylim(0, 4000)
axes[0].set_ylabel("k-distance co trong so (m)")
axes[-1].legend(fontsize=7, loc="upper left")
fig.suptitle(f"DBSCAN: k-distance co trong so - {SAMPLE_ID} (net dut = min_samples theo spec; vach xam = eps trong config)")
fig.tight_layout()
fig_path = resolve_path(cfg, "figures_dir") / f"dbscan_kdist_{SAMPLE_ID}.png"
fig.savefig(fig_path, dpi=150)
print("Da luu:", fig_path.relative_to(cfg["_root"]))

### Eps ứng viên

Gom các điểm gãy (làm tròn tới 50 m) theo từng k; dùng làm lưới eps mịn cho mục 4.3, cộng thêm lưới thô trong config.

In [ ]:
knees = kdist.dropna(subset=["knee_eps_m"]).assign(knee_r=lambda d: (d.knee_eps_m / 50).round() * 50)
print("Diem gay (m) theo k va lat cat:")
print(knees.pivot_table(index="k_label", columns="time_slice", values="knee_r", aggfunc="first").to_string())
cand = sorted(set(knees.knee_r.astype(int)) | set(DB["eps_m"]))
print("\nEps ung vien cho 4.3:", cand)

## 4.2 So sánh chạy trên từng chuyến (`full`) với chạy có trọng số (`weighted`)

- Lát cắt WEEKEND (lớn nhất), mẫu con tăng dần 10k → 20k → toàn bộ, lấy ngẫu nhiên có seed.
- 3 cấu hình đại diện: (eps 500 m, 0,5%), (eps 1000 m, 0,5%), (eps 1500 m, spec 20). `min_samples` theo tỷ lệ tính trên cỡ mẫu con.
- **Bảo vệ RAM:** trước mỗi lần chạy `full`, ước tính số cặp láng giềng bằng `estimate_full_neighbors` (đã kiểm tra khớp chính xác với số đếm thật). Bỏ qua lần chạy nếu ước tính vượt `MAX_FULL_GB`, hoặc nếu RAM hệ thống đã vượt `ram_stop_ratio` (85%) trong config.
- So sánh: ARI giữa hai cách, số cụm, Noise Ratio, tập điểm nhiễu có trùng khớp không; runtime và RAM thêm (`mem_delta_mb`) đo bằng `measure`.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score
from src.evaluation import measure, system_ram_ratio
from src.models.dbscan import estimate_full_neighbors, fit_weighted
from src.m2_pilot import subsample

MAX_FULL_GB = 2.0
SIZES = [10000, 20000, None]          # None = toan bo lat cat
CONFIGS = [(500, ("frac", 0.005)), (1000, ("frac", 0.005)), (1500, ("spec", 20))]
part_all = df[df.time_slice == "WEEKEND"].reset_index(drop=True)

rows = []
for n in SIZES:
    part = subsample(part_all, n if n else len(part_all), seed=cfg["project"]["seed"])
    X = part[["x_m", "y_m"]].to_numpy("float64")
    for eps, (kind, v) in CONFIGS:
        ms = min_samples_from_frac(len(X), v) if kind == "frac" else v
        r = {"n": len(X), "eps_m": eps, "min_samples": ms, "ms_rule": f"{v * 100:g}%" if kind == "frac" else f"spec {v}"}
        pairs = estimate_full_neighbors(X, eps)
        r["est_pairs"] = pairs; r["est_gb"] = pairs * 8 / 1e9
        (lw, dbw), r["weighted_sec"], _, r["weighted_mem_mb"] = measure(lambda: fit_weighted(X, eps, ms))
        ram = system_ram_ratio()
        if r["est_gb"] > MAX_FULL_GB or ram > cfg["evaluation"]["ram_stop_ratio"]:
            r["full_status"] = f"bo qua (uoc tinh {r['est_gb']:.1f} GB, RAM he thong {ram:.0%})"
        else:
            lf, r["full_sec"], _, r["full_mem_mb"] = measure(lambda: DBSCAN(eps=eps, min_samples=ms).fit_predict(X))
            r.update(full_status="da chay", ari=adjusted_rand_score(lf, lw),
                     same_noise=bool(np.array_equal(lf == -1, lw == -1)),
                     n_clusters_full=len(set(lf) - {-1}), noise_full=float((lf == -1).mean()))
        r.update(n_clusters_weighted=len(set(lw) - {-1}), noise_weighted=float((lw == -1).mean()))
        rows.append(r)
        print(f"n={len(X):>6,} eps={eps:>4} ms={ms:>4}: {r['full_status']:<40s} "
              f"full {r.get('full_sec', float('nan')):6.2f}s / {r.get('full_mem_mb', float('nan')):7.1f} MB | "
              f"weighted {r['weighted_sec']:.3f}s / {r['weighted_mem_mb']:.1f} MB | ARI {r.get('ari', float('nan')):.4f}")
cmp42 = pd.DataFrame(rows).assign(time_slice="WEEKEND", sample_id=SAMPLE_ID)
p42 = resolve_path(cfg, "run_log").parent / f"dbscan_full_vs_weighted_{SAMPLE_ID}.csv"
cmp42.to_csv(p42, index=False)
print("Da luu:", p42.relative_to(cfg["_root"]))

In [ ]:
ran = cmp42[cmp42.full_status == "da chay"]
print("Tat ca lan chay full: ARI = 1 va tap nhieu trung khop:", bool((ran.ari == 1).all() and ran.same_noise.all()),
      f"({len(ran)}/{len(cmp42)} lan chay full)")
cmp42[["n", "eps_m", "min_samples", "ms_rule", "est_gb", "full_status", "full_sec", "full_mem_mb",
       "weighted_sec", "weighted_mem_mb", "ari", "n_clusters_full", "n_clusters_weighted", "noise_full", "noise_weighted"]].round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for eps, (kind, v) in CONFIGS:
    d = cmp42[cmp42.eps_m == eps]
    lab = f"eps {eps}, {d.ms_rule.iloc[0]}"
    l, = axes[0].plot(d.n, d.full_sec, "o-", label=f"full - {lab}")
    axes[0].plot(d.n, d.weighted_sec, "s--", color=l.get_color(), label=f"weighted - {lab}")
    axes[1].plot(d.n, d.full_mem_mb, "o-", color=l.get_color(), label=f"full - {lab}")
    axes[1].plot(d.n, d.est_gb * 1000, ":", color=l.get_color(), alpha=.6, label=f"uoc tinh chi so - {lab}")
axes[0].set_yscale("log"); axes[0].set_ylabel("Thoi gian fit (s, thang log)")
axes[1].set_ylabel("RAM them khi fit (MB)")
for ax in axes:
    ax.set_xlabel("So chuyen (WEEKEND)"); ax.grid(alpha=.3); ax.legend(fontsize=7)
fig.suptitle(f"DBSCAN: chay tren tung chuyen (full) vs toa do co trong so (weighted) - {SAMPLE_ID}")
fig.tight_layout()
fig.savefig(resolve_path(cfg, "figures_dir") / f"dbscan_full_vs_weighted_{SAMPLE_ID}.png", dpi=150)